In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# INN Hotels Project

## Context

A significant number of hotel bookings are called-off due to cancellations or no-shows. The typical reasons for cancellations include change of plans, scheduling conflicts, etc. This is often made easier by the option to do so free of charge or preferably at a low cost which is beneficial to hotel guests but it is a less desirable and possibly revenue-diminishing factor for hotels to deal with. Such losses are particularly high on last-minute cancellations. 

The new technologies involving online booking channels have dramatically changed customers’ booking possibilities and behavior. This adds a further dimension to the challenge of how hotels handle cancellations, which are no longer limited to traditional booking and guest characteristics. 

The cancellation of bookings impact a hotel on various fronts:
* Loss of resources (revenue) when the hotel cannot resell the room.
* Additional costs of distribution channels by increasing commissions or paying for publicity to help sell these rooms.
* Lowering prices last minute, so the hotel can resell a room, resulting in reducing the profit margin.
* Human resources to make arrangements for the guests.

## Objective
The increasing number of cancellations calls for a Machine Learning based solution that can help in predicting which booking is likely to be canceled. INN Hotels Group has a chain of hotels in Portugal, they are facing problems with the high number of booking cancellations and have reached out to your firm for data-driven solutions. You as a data scientist have to analyze the data provided to find which factors have a high influence on booking cancellations, build a predictive model that can predict which booking is going to be canceled in advance, and help in formulating profitable policies for cancellations and refunds.

* Predict if a booking is going to be canceled in advance
* Create a cancellation policy that will increase profitability based on our predictions

## Data Description
The data contains the different attributes of customers' booking details. The detailed data dictionary is given below.


**Data Dictionary**

* Booking_ID: unique identifier of each booking
* no_of_adults: Number of adults
* no_of_children: Number of Children
* no_of_weekend_nights: Number of weekend nights (Saturday or Sunday) the guest stayed or booked to stay at the hotel
* no_of_week_nights: Number of week nights (Monday to Friday) the guest stayed or booked to stay at the hotel
* type_of_meal_plan: Type of meal plan booked by the customer:
    * Not Selected – No meal plan selected
    * Meal Plan 1 – Breakfast
    * Meal Plan 2 – Half board (breakfast and one other meal)
    * Meal Plan 3 – Full board (breakfast, lunch, and dinner)
* required_car_parking_space: Does the customer require a car parking space? (0 - No, 1- Yes)
* room_type_reserved: Type of room reserved by the customer. The values are ciphered (encoded) by INN Hotels.
* lead_time: Number of days between the date of booking and the arrival date
* arrival_year: Year of arrival date
* arrival_month: Month of arrival date
* arrival_date: Date of the month
* market_segment_type: Market segment designation.
* repeated_guest: Is the customer a repeated guest? (0 - No, 1- Yes)
* no_of_previous_cancellations: Number of previous bookings that were canceled by the customer prior to the current booking
* no_of_previous_bookings_not_canceled: Number of previous bookings not canceled by the customer prior to the current booking
* avg_price_per_room: Average price per day of the reservation; prices of the rooms are dynamic. (in euros)
* no_of_special_requests: Total number of special requests made by the customer (e.g. high floor, view from the room, etc)
* booking_status: Flag indicating if the booking was canceled or not.

In [ ]:
# Library to suppress warnings
import warnings

warnings.filterwarnings("ignore")

# Import Numpy & Pandas
import pandas as pd
import numpy as np

# Display max rows and columns
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

# Library to split data into train & test
from sklearn.model_selection import train_test_split

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Decision Tree Classifiers
from sklearn.tree import DecisionTreeClassifier
from sklearn import tree

# Tuning
from sklearn.model_selection import GridSearchCV

# To perform statistical analysis
import scipy.stats as stats

# Metrics Scores
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    recall_score,
    precision_score,
    confusion_matrix,
    roc_auc_score,
    plot_confusion_matrix,
    precision_recall_curve,
    roc_curve,
    make_scorer,
)

In [ ]:
HG_df = pd.read_csv('../input/inn-hotel-booking-information/INNHotelsGroup.csv')
print(HG_df.shape)
HG_df.head(10)

In [ ]:
HG_df.info()

### EDA

In [ ]:
# Total guest in attendance based on "Not_Canceled" bookings

# Create Temporary df for months & total guests
Months = np.arange(1,13)
total_guests_data = []

# iterate through the Months df to group total guests to each month
for i in Months:
    HG_m = HG_df[(HG_df.arrival_month == i) & (HG_df.booking_status == "Not_Canceled")]
    total_guests = sum(HG_m.no_of_adults) + sum(HG_m.no_of_children)
    total_guests_data.append({"Month": i, "Total_Guests": total_guests})
    i + 1
total_guests_df = pd.DataFrame(total_guests_data)

# Create bar plot
total_guests_df.plot(x="Month", y="Total_Guests", kind="bar")
plt.show()

In [ ]:
total_cancelled_booking_sr_data = []
annotations = []
special_request = np.arange(0,6)

# iterate through the Months df to group total guests to each month
for i in special_request:
    HG_sr_c = HG_df[
        (HG_df.no_of_special_requests == i) & (HG_df.booking_status == "Canceled")
    ]
    HG_sr = HG_df[HG_df.no_of_special_requests == i]
    # Calculate percentage of special requests that cancelled
    perc_Calculation = round(
        HG_sr_c.no_of_special_requests.count() / HG_sr.no_of_special_requests.count(), 2
    )
    # Append data to prep for df
    total_cancelled_booking_sr_data.append(
        {"number_of_special_request": i, "perc_of_cancelation": perc_Calculation}
    )
    annotations.append(perc_Calculation)
    i + 1

    # Create df
total_cancelled_booking_sr_df = pd.DataFrame(total_cancelled_booking_sr_data)

sns.scatterplot(
    data=total_cancelled_booking_sr_df,
    x="number_of_special_request",
    y="perc_of_cancelation",
)
plt.title("ScatterPlot showing Special Request Cancelation trend")
for i, label in enumerate(annotations):
    plt.annotate(
        label,
        (
            total_cancelled_booking_sr_df.number_of_special_request[i],
            total_cancelled_booking_sr_df.perc_of_cancelation[i],
        ),
    )

plt.show()

Observation:
* 43% of cancellations happened when guest had 0 special requests during their booking process. This percentage became incrementally smaller as more special requests were filed.
* Adding in proactive options for special requests during the booking process may help reduce the chance of a guest cancelling their booking.

### Data PreProcessing

In [ ]:
# Create a copy for Data Preprocessing
HG_df_dpp = HG_df.copy()
# Drop Booking_ID as it is not needed for analysis
HG_df_dpp = HG_df_dpp.drop("Booking_ID", axis=1)

# Convert object Dtypes to Category
columns_o = HG_df_dpp.select_dtypes(include=["object"]).columns.tolist()
for colname in columns_o:
    HG_df_dpp[colname] = HG_df_dpp[colname].astype("category")

# Gather info to get a better understand of the dataset
HG_df_dpp.info()
print("*" * 50)
print("*" * 50)

# Iterate through the category columns, convert to a list, and print the value counts of each.
columns_c = HG_df_dpp.select_dtypes(include=["category"]).columns.tolist()
for i in columns_c:
    print(f"Value Count for {i}")
    print(HG_df_dpp[i].value_counts())
    print(f"To verify total counts {HG_df_dpp[i].count()}")
    print("*" * 50)

Observations:
* I do not see any outlandish categories or null values, so I will move on to the next step.

In [ ]:
# Convert Year, Month, and Day to a Date Integer for processing
HG_df_dpp["dateInt"] = (
    HG_df_dpp["arrival_year"].astype(str)
    + "-"
    + HG_df_dpp["arrival_month"].astype(str).str.zfill(2)
    + "-"
    + HG_df_dpp["arrival_date"].astype(str).str.zfill(2)
)

# Convert the DateInt Colum to a datetime dtype
HG_df_dpp = HG_df_dpp[HG_df_dpp["dateInt"] != "2018-02-29"]
HG_df_dpp["Date"] = pd.to_datetime(HG_df_dpp["dateInt"], format="%Y-%m-%d")
HG_df_dpp = HG_df_dpp.drop("dateInt", axis=1)

#Convert Date to day of week
HG_df_dpp["DayofWeek"] = HG_df_dpp["Date"].dt.day_name().astype("category")
HG_df_dpp = HG_df_dpp.drop("arrival_date", axis=1)
HG_df_dpp.info()

In [ ]:
# Reinitialize the new numeric columns
numeric_columns = HG_df_dpp.select_dtypes(include=np.number).columns.tolist()
# let's plot the boxplots of all columns to check for outliers
plt.figure(figsize=(20, 30))

for i, variable in enumerate(numeric_columns):
    plt.subplot(5, 4, i + 1)
    plt.boxplot(HG_df_dpp[variable], whis=1.5)
    plt.tight_layout()
    plt.title(variable)

plt.show()

Observations:
* While checking for outliers, I decided that these outliers for necessary for analysis as the data points could not be easily jusitified removal or scaling.

In [ ]:
columns_c = HG_df_dpp.select_dtypes(include=["int64"]).columns.tolist()
for i in columns_c:
    print(f"Value Count for {i}")
    print(HG_df_dpp[i].value_counts())
    print(f"To verify total counts {HG_df_dpp[i].count()}")
    print("*" * 50)

Observations:
* Many categorical types for int64, move forward with grouping

In [ ]:
# can add custom labels
HG_df_dpp["lead_time_bin"] = pd.cut(
    HG_df_dpp["lead_time"],
    [-np.inf, 30, 90, 180, 365, np.inf],
    labels=[
        "30_days",
        "30_to_90_days",
        "90_days_to_half_year",
        "half_year_to_year",
        "more_than_year",
    ],
)
HG_df_dpp.drop(["lead_time"], axis=1, inplace=True)
HG_df_dpp["lead_time_bin"].value_counts(dropna=False)

In [ ]:
# can add custom labels
HG_df_dpp["no_of_children_bin"] = pd.cut(
    HG_df_dpp["no_of_children"],
    [-np.inf, 0, 1, 2, 3, np.inf],
    labels=["0", "1", "2", "3", "4 or more"],
)
HG_df_dpp.drop(["no_of_children"], axis=1, inplace=True)
HG_df_dpp["no_of_children_bin"].value_counts(dropna=False)

In [ ]:
# can add custom labels
HG_df_dpp["no_of_previous_bookings_not_canceled_bin"] = pd.cut(
    HG_df_dpp["no_of_previous_bookings_not_canceled"],
    [-np.inf, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, np.inf],
    labels=["0", "1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11 or more"],
)
HG_df_dpp.drop(["no_of_previous_bookings_not_canceled"], axis=1, inplace=True)
HG_df_dpp["no_of_previous_bookings_not_canceled_bin"].value_counts(dropna=False)

In [ ]:
# can add custom labels
HG_df_dpp["no_of_previous_cancellations_bin"] = pd.cut(
    HG_df_dpp["no_of_previous_cancellations"],
    [-np.inf, 0, 1, 2, 3, 4, 5, np.inf],
    labels=["0", "1", "2", "3", "4", "5", "6 or more"],
)
HG_df_dpp.drop(["no_of_previous_cancellations"], axis=1, inplace=True)
HG_df_dpp["no_of_previous_cancellations_bin"].value_counts(dropna=False)

In [ ]:
HG_df_dpp.info()

Observations:
* I simplified the dataset and redue total memory usage from 4.3 to 3.4MB

In [ ]:
# import preprocessing library to encode booking status
from sklearn.preprocessing import LabelEncoder

# transform the booking status to binary
le = LabelEncoder()
booking_status_encoded = le.fit_transform(HG_df_dpp["booking_status"])

HG_df_dpp["booking_status_encoded"] = booking_status_encoded

### Continued EDA

In [ ]:
plt.figure(figsize=(24, 16))
sns.lineplot(
    data=HG_df_dpp[HG_df_dpp["Date"] > "01-01-2018"],
    x="Date",
    y="avg_price_per_room",
    hue="booking_status",
)
plt.show()

Observation:
* The primary spikes for cancellation seem to happen when the average price per room is either higher or lower than that of the non cancelled rooms. This could indicate that if a room is priced to aggressively or at a premium, the likelyhood of a cancelled booking increases.

# Building a Decision Tree

In [ ]:
# defining X and y variables
drop_var = ["booking_status", "Date", "booking_status_encoded"]
X = HG_df_dpp.drop(drop_var, axis="columns")
y = HG_df_dpp["booking_status_encoded"]

# Create dummy variables for object and category dtypes
X = pd.get_dummies(
    X,
    columns=X.select_dtypes(include=["object", "category"]).columns.tolist(),
    drop_first=True,
)
X.info()

In [ ]:
# split X and y into training and testing datasets split into .60 to .40 size
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=1)

print("Number of rows in train data =", X_train.shape[0])
print("Number of rows in test data =", X_test.shape[0])
print('*' * 50)
print("Percentage of classes in training set:")
print(y_train.value_counts(normalize=True))
print('*' * 50)
print("Percentage of classes in test set:")
print(y_test.value_counts(normalize=True))

In [ ]:
# fit a decision tree model using the gini critera
model = DecisionTreeClassifier(criterion="gini", random_state=1)
model.fit(X_train, y_train)

In [ ]:
# defining a function to compute different metrics to check performance of a classification model built using statsmodels
def model_performance_classification_statsmodels(
    model, predictors, target, threshold=0.5
):
    """
    Function to compute different metrics to check classification model performance

    model: classifier
    predictors: independent variables
    target: dependent variable
    threshold: threshold for classifying the observation as class 1
    """

    # checking which probabilities are greater than threshold
    pred_temp = model.predict(predictors) > threshold
    # rounding off the above values to get classes
    pred = np.round(pred_temp)

    acc = accuracy_score(target, pred)  # to compute Accuracy
    recall = recall_score(target, pred)  # to compute Recall
    precision = precision_score(target, pred)  # to compute Precision
    f1 = f1_score(target, pred)  # to compute F1-score

    # creating a dataframe of metrics
    df_perf = pd.DataFrame(
        {"Accuracy": acc, "Recall": recall, "Precision": precision, "F1": f1,},
        index=[0],
    )

    return df_perf

# defining a function to plot the confusion_matrix of a classification model


def confusion_matrix_statsmodels(model, predictors, target, threshold=0.5):
    """
    To plot the confusion_matrix with percentages

    model: classifier
    predictors: independent variables
    target: dependent variable
    threshold: threshold for classifying the observation as class 1
    """
    y_pred = model.predict(predictors) > threshold
    cm = confusion_matrix(target, y_pred)
    labels = np.asarray(
        [
            ["{0:0.0f}".format(item) + "\n{0:.2%}".format(item / cm.flatten().sum())]
            for item in cm.flatten()
        ]
    ).reshape(2, 2)

    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=labels, fmt="")
    plt.ylabel("True label")
    plt.xlabel("Predicted label")

# create a function to evaluate models using model_performance_classification_statsmodel & confusion_matrix_statsmodels
def performance_Matrix(
    Model_type,
    Model,
    X_training_data,
    y_training_data,
    X_testing_data,
    y_testing_data,
    threshold=0.5,
):
    """
    Model_type = Model Label using string format
    Model = Fitted Model that you want to pass the data through
    X_train_data = X Training Data
    y_train_data = Y Training Data
    X_testing_data= X Testing Data
    y_testing_data = Y Testing Data    
    """

    print(f"Training Performance for model type {Model_type}:")
    print(
        model_performance_classification_statsmodels(
            Model, X_training_data, y_training_data, threshold=threshold
        )
    )
    print("*" * 50)
    print(f"Test Perfromancefor model type {Model_type}")
    print(
        model_performance_classification_statsmodels(
            Model, X_testing_data, y_test, threshold=threshold
        )
    )
    print("*" * 50)
    print("*" * 50)
    print("Confusion Matrix for training & testing")
    confusion_matrix_statsmodels(
        Model, X_training_data, y_training_data, threshold=threshold
    )
    print("*" * 50)
    confusion_matrix_statsmodels(
        Model, X_testing_data, y_testing_data, threshold=threshold
    )

In [ ]:
performance_Matrix("Gini", model, X_train, y_train, X_test, y_test)

Observations:
* As expected, the training data 's f1 score is very close to 1 while the testing data is nearing .90. While the model is capturing a good amount of test data, we can assume without tuning this model is overfitting.

In [ ]:
column_names = list(X.columns)
feature_names = column_names

plt.figure(figsize=(20, 30))

out = tree.plot_tree(
    model,
    feature_names=feature_names,
    filled=True,
    fontsize=9,
    node_ids=True,
    class_names=True,
)
for o in out:
    arrow = o.arrow_patch
    if arrow is not None:
        arrow.set_edgecolor("black")
        arrow.set_linewidth(1)
plt.show()

Observation:
* Due to the overfitting, the decision tree branchs are very complex. Should look at limiting depth and post pruning

In [ ]:
# Gother the model's important variables and sort by importance
importances = model.feature_importances_
indices = np.argsort(importances)

# Plot importance variables
plt.figure(figsize=(12, 12))
plt.title("Feature Importances")
plt.barh(range(len(indices)), importances[indices], color="violet", align="center")
plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
plt.xlabel("Relative Importance")
plt.show()

Observation:
* The average price per room, the lead time inbetween 6 to 12 months, the arrival month, online booking, and number of special request have a high level of importance when determining if a guest will cancel.


## GridSearch for Hyperparameter tuning

In [ ]:
# Choose the type of classifier.
estimator = DecisionTreeClassifier(random_state=1)

# Grid of parameters to choose from

parameters = {
    "max_depth": [5, 10, 15],
    "criterion": ["entropy", "gini"],
    "splitter": ["best", "random"],
    "min_impurity_decrease": [0.000001, 0.00001],
}

# Type of scoring used to compare parameter combinations
# Based on business needs, I decided to use F1_score as the metric
acc_scorer = make_scorer(f1_score)

# Run the grid search
grid_obj = GridSearchCV(estimator, parameters, scoring=acc_scorer, cv=5)
grid_obj = grid_obj.fit(X_train, y_train)

# Set the clf to the best combination of parameters
estimator = grid_obj.best_estimator_

# Fit the best algorithm to the data.
estimator.fit(X_train, y_train)

In [ ]:
performance_Matrix("Entropy", estimator, X_train, y_train, X_test, y_test)

## Observation
* By reduce the number of branches, I was able to better generalize the testing data which allowed for a higher f1 score.
* Before prepruning, I was able to accurately categorize 85% of the testing data set.

Will Post Pruning help increase the f1_score of the test data set?

In [ ]:
clf = DecisionTreeClassifier(random_state=1)
path = clf.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas, impurities = path.ccp_alphas, path.impurities

In [ ]:
fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(ccp_alphas[:-1], impurities[:-1], marker="o", drawstyle="steps-post")
ax.set_xlabel("effective alpha")
ax.set_ylabel("total impurity of leaves")
ax.set_title("Total Impurity vs effective alpha for training set")
plt.show()

In [ ]:
clfs = []
for ccp_alpha in ccp_alphas:
    clf = DecisionTreeClassifier(random_state=1, ccp_alpha=ccp_alpha)
    clf.fit(X_train, y_train)
    clfs.append(clf)
print(
    "Number of nodes in the last tree is: {} with ccp_alpha: {}".format(
        clfs[-1].tree_.node_count, ccp_alphas[-1]
    )
)

In [ ]:
clfs = clfs[:-1]
ccp_alphas = ccp_alphas[:-1]

node_counts = [clf.tree_.node_count for clf in clfs]
depth = [clf.tree_.max_depth for clf in clfs]
fig, ax = plt.subplots(2, 1, figsize=(10, 7))
ax[0].plot(ccp_alphas, node_counts, marker="o", drawstyle="steps-post")
ax[0].set_xlabel("alpha")
ax[0].set_ylabel("number of nodes")
ax[0].set_title("Number of nodes vs alpha")
ax[1].plot(ccp_alphas, depth, marker="o", drawstyle="steps-post")
ax[1].set_xlabel("alpha")
ax[1].set_ylabel("depth of tree")
ax[1].set_title("Depth vs alpha")
fig.tight_layout()

In [ ]:
f1_train = []
for clf in clfs:
    pred_train = clf.predict(X_train)
    values_train = f1_score(y_train, pred_train)
    f1_train.append(values_train)

In [ ]:
f1_test = []
for clf in clfs:
    pred_test = clf.predict(X_test)
    values_test = f1_score(y_test, pred_test)
    f1_test.append(values_test)

In [ ]:
# Plot F1_score & Alpha points to visualize the optimal alpha
fig, ax = plt.subplots(figsize=(15, 5))
ax.set_xlabel("alpha")
ax.set_ylabel("F1 S")
ax.set_title("F1 Score vs alpha for training and testing sets")
ax.plot(ccp_alphas, f1_train, marker="o", label="train", drawstyle="steps-post")
ax.plot(ccp_alphas, f1_test, marker="o", label="test", drawstyle="steps-post")
ax.legend()
plt.show()

In [ ]:
# creating the model where we get highest train and test f1_score
index_best_model = np.argmax(f1_test)
best_model = clfs[index_best_model]
print(best_model)

In [ ]:
performance_Matrix("Gini", best_model, X_train, y_train, X_test, y_test)

In [ ]:
plt.figure(figsize=(10, 10))

out = tree.plot_tree(
    best_model,
    feature_names=feature_names,
    filled=True,
    fontsize=9,
    node_ids=True,
    class_names=True,
)
for o in out:
    arrow = o.arrow_patch
    if arrow is not None:
        arrow.set_edgecolor("black")
        arrow.set_linewidth(1)
plt.show()

In [ ]:
importances = best_model.feature_importances_
indices = np.argsort(importances)

plt.figure(figsize=(12, 12))
plt.title("Feature Importances")
plt.barh(range(len(indices)), importances[indices], color="violet", align="center")
plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
plt.xlabel("Relative Importance")
plt.show()

## Observations
* Like the original model, the same features are relatively important but a notice change in order. 6 months to a year lead time has overtaken the average price of the room while online bookings move up by several places.
* I can assume that lead times play a big factor in whether a person will cancel their booking

In [ ]:
decision_tree_perf_train = model_performance_classification_statsmodels(
    model, X_train, y_train
)

decision_tree_tune_perf_train = model_performance_classification_statsmodels(
    estimator, X_train, y_train
)

decision_tree_postpruned_perf_train = model_performance_classification_statsmodels(
    best_model, X_train, y_train
)

# training performance comparison

models_train_comp_df = pd.concat(
    [
        decision_tree_perf_train.T,
        decision_tree_tune_perf_train.T,
        decision_tree_postpruned_perf_train.T,
    ],
    axis=1,
)
models_train_comp_df.columns = [
    "Decision Tree sklearn",
    "Decision Tree (Pre-Pruning)",
    "Decision Tree (Post-Pruning)",
]
print("Training performance comparison:")
models_train_comp_df

In [ ]:
decision_tree_perf_test = model_performance_classification_statsmodels(
    model, X_test, y_test
)

decision_tree_tune_perf_test = model_performance_classification_statsmodels(
    estimator, X_test, y_test
)

decision_tree_postpruned_perf_test = model_performance_classification_statsmodels(
    best_model, X_test, y_test
)

# training performance comparison

models_test_comp_df = pd.concat(
    [
        decision_tree_perf_test.T,
        decision_tree_tune_perf_test.T,
        decision_tree_postpruned_perf_test.T,
    ],
    axis=1,
)
models_test_comp_df.columns = [
    "Decision Tree sklearn",
    "Decision Tree (Pre-Pruning)",
    "Decision Tree (Post-Pruning)",
]
print("Training performance comparison:")
models_test_comp_df

Observation:
* The post-pruning model, also known as model "best_model" shows the highest F1 score for testing data. I would use this model for future predictions.

Recommendations:
* Knowing that lead time, price, and online bookings have the highest influence on cancellations we can infer that having cancellation clause during the online booking process would influence how customer's book.
* Keep price of rooms near competitive pricing as it seems like guest will be book a room with the expectation of continued searches.
* Since repeating_guests have a very low cancellation rate, creating a loyalty program for those guest would help incentives other to move into that category.
* During the online booking process, offering addtional customizations or special request would help reduce the likelihood of a cancelled booking.